# **02. Data Cleaning, NLP Preprocessing, and Feature Engineering**

This notebook prepares the multilingual support-ticket dataset for modelling. It documents how `subject` and `body` are combined with safe metadata while excluding response fields that would cause target leakage.

# **1. Environment Setup**

We use the same reusable project functions as the training pipeline. This keeps notebook decisions aligned with the implementation in `src/`.

In [1]:
# Import libraries and project helper functions.
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src' / 'customer_support_ai').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from customer_support_ai.config import COMPATIBLE_DATASET_FILENAMES, COMPATIBLE_DATASET_PATHS, SAFE_METADATA_CANDIDATES
from customer_support_ai.data_preprocessing import build_modelling_frame, identify_leakage_columns, load_project_dataset, normalise_columns
from customer_support_ai.features import clean_text, metadata_to_tokens

# **2. Load and Standardise Dataset**

The compatible CSV names are shown explicitly so anyone running the notebook knows which Kaggle files are expected.

In [2]:
# Load the compatible multilingual ticket datasets.
print('Compatible CSV files:')
for filename in COMPATIBLE_DATASET_FILENAMES:
    print(f'- {filename}')

raw = load_project_dataset(COMPATIBLE_DATASET_PATHS)
data = normalise_columns(raw)

print(f"Raw rows before deduplication: {raw.attrs['raw_rows_before_dedup']:,}")
print(f"Duplicate rows removed: {raw.attrs['duplicate_rows_removed']:,}")
print(f'Rows: {len(data):,}')
print(f'Columns after normalisation: {data.shape[1]}')
data.head()

Compatible CSV files:
- aa_dataset-tickets-multi-lang-5-2-50-version.csv
- dataset-tickets-multi-lang-4-20k.csv
- dataset-tickets-multi-lang3-4k.csv


Raw rows before deduplication: 52,587
Duplicate rows removed: 8,309
Rows: 44,278
Columns after normalisation: 19


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,source_file,business_type,tag_9
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN,aa_dataset-tickets-multi-lang-5-2-50-version.csv,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN,aa_dataset-tickets-multi-lang-5-2-50-version.csv,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN,aa_dataset-tickets-multi-lang-5-2-50-version.csv,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN,aa_dataset-tickets-multi-lang-5-2-50-version.csv,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN,aa_dataset-tickets-multi-lang-5-2-50-version.csv,NaN,NaN


# **3. Feature Safety Decisions**

The initial prediction should use the customer request and metadata available at ticket creation time. The `answer` field is excluded because it is written after support staff respond.

In [3]:
# Separate safe modelling fields from private or leakage-prone fields.
leakage_columns = identify_leakage_columns(data)
private_identifier_columns = []
available_safe_metadata = [col for col in SAFE_METADATA_CANDIDATES if col in data.columns]

feature_decisions = pd.DataFrame([
    {'field_group': 'Text input', 'columns': 'subject + body', 'decision': 'Use as main NLP input'},
    {'field_group': 'Safe metadata', 'columns': ', '.join(available_safe_metadata), 'decision': 'Use as metadata tokens'},
    {'field_group': 'Leakage risk', 'columns': ', '.join(leakage_columns), 'decision': 'Exclude because values are known after support response'},
    {'field_group': 'Targets', 'columns': 'queue, priority', 'decision': 'Predict queue/routing and priority'},
])

feature_decisions

,field_group,columns,decision
0,Text input,subject + body,Use as main NLP input
1,Safe metadata,"type, language, tag_1, tag_2, tag_3, tag_4, ta...",Use as metadata tokens
2,Leakage risk,answer,Exclude because values are known after support...
3,Targets,"queue, priority",Predict queue/routing and priority


# **4. Text Cleaning Demonstration**

The cleaning function lowercases text, masks obvious personal data, removes punctuation noise, and keeps important support terms.

In [4]:
# Demonstrate the cleaning step on a realistic ticket sentence.
example_ticket = 'URGENT: VPN access is down for the Berlin office. Email me at customer@example.com.'

print('Original:', example_ticket)
print('Cleaned :', clean_text(example_ticket))

Original: URGENT: VPN access is down for the Berlin office. Email me at customer@example.com.
Cleaned : urgent vpn access is down for the berlin office email me at emailaddress


# **5. Metadata Tokenisation**

Metadata such as ticket type, language, and tags are converted into readable tokens for the TF-IDF pipeline.

In [5]:
# Convert one row of safe metadata into text tokens.
metadata_example = data[available_safe_metadata].fillna('unknown').iloc[0]
metadata_to_tokens(metadata_example)

'type_incident language_de tag_1_security tag_2_outage tag_3_disruption tag_4_data_breach tag_5_unknown tag_6_unknown tag_7_unknown tag_8_unknown tag_9_unknown'

# **6. Build the Model-Ready Frame**

The reusable preprocessing function combines `subject + body`, cleans the text, adds safe metadata tokens, and returns the target columns.

In [6]:
# Build the final modelling frame used by the training pipeline.
frame, queue_col, priority_col = build_modelling_frame(data)

print(f'Model rows: {len(frame):,}')
print(f'Queue target: {queue_col}')
print(f'Priority target: {priority_col}')
print(f'Text columns: subject={frame.attrs.get("subject_column")}, body={frame.attrs.get("text_column")}')
print(f'Safe metadata used: {frame.attrs.get("metadata_columns")}')
print(f'Leakage columns excluded: {frame.attrs.get("leakage_columns")}')

frame[['ticket_text', 'clean_text', 'model_text', queue_col, priority_col]].head()

Model rows: 44,275
Queue target: queue
Priority target: priority
Text columns: subject=subject, body=body
Safe metadata used: ['type', 'language', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8', 'tag_9']
Leakage columns excluded: ['answer']


,ticket_text,clean_text,model_text,queue,priority
0,Wesentlicher Sicherheitsvorfall Sehr geehrtes ...,wesentlicher sicherheitsvorfall sehr geehrtes ...,wesentlicher sicherheitsvorfall sehr geehrtes ...,Technical Support,high
1,"Account Disruption Dear Customer Support Team,...",account disruption dear customer support team ...,account disruption dear customer support team ...,Technical Support,high
2,Query About Smart Home System Integration Feat...,query about smart home system integration feat...,query about smart home system integration feat...,Returns and Exchanges,medium
3,Inquiry Regarding Invoice Details Dear Custome...,inquiry regarding invoice details dear custome...,inquiry regarding invoice details dear custome...,Billing and Payments,low
4,Question About Marketing Agency Software Compa...,question about marketing agency software compa...,question about marketing agency software compa...,Sales and Pre-Sales,medium


# **7. Processed Data Quality Check**

We confirm the processed dataset has complete targets and a rich number of unique model inputs.

In [7]:
# Check model-ready text and target completeness.
processed_quality = pd.Series({
    'rows': len(frame),
    'empty_model_text_rows': (frame['model_text'].str.len() == 0).sum(),
    'missing_queue_rows': frame[queue_col].isna().sum(),
    'missing_priority_rows': frame[priority_col].isna().sum(),
    'unique_clean_text_values': frame['clean_text'].nunique(),
    'unique_model_text_values': frame['model_text'].nunique(),
})

processed_quality

rows                        44275
empty_model_text_rows           0
missing_queue_rows              0
missing_priority_rows           0
unique_clean_text_values    44273
unique_model_text_values    44273
dtype: int64

# **8. Label Signal Check**

A useful dataset should show some relationship between words/tags and targets. We inspect top tags by queue as a simple feature-signal check.

In [8]:
# Show the most common first tag for each queue.
tag_queue_table = (
    data.groupby([queue_col, 'tag_1'])
    .size()
    .rename('count')
    .reset_index()
    .sort_values([queue_col, 'count'], ascending=[True, False])
    .groupby(queue_col)
    .head(3)
)

tag_queue_table

,queue,tag_1,count
4,Billing and Payments,Billing,2064
41,Billing and Payments,Security,664
17,Billing and Payments,Feedback,349
125,Customer Service,Security,1200
86,Customer Service,Feedback,930
136,Customer Service,Technical,747
179,General Inquiry,Security,110
163,General Inquiry,Feedback,101
162,General Inquiry,Feature,83
227,Human Resources,Security,195


# **9. Save Processed Dataset for Reuse**

Saving the processed dataset makes the workflow easier to inspect and supports later notebooks.

In [9]:
# Save model-ready data for later notebooks and report checks.
processed_path = PROJECT_ROOT / 'data' / 'processed' / 'model_ready_tickets.csv'
processed_path.parent.mkdir(parents=True, exist_ok=True)
frame.to_csv(processed_path, index=False)

print(f'Saved processed data to: {processed_path}')

Saved processed data to: C:\Users\taals\OneDrive\Documents\New project 2\data\processed\model_ready_tickets.csv


# **10. Cleaning and Feature Engineering Findings**

This cell summarises the preprocessing decisions for the report.

In [10]:
# Summarise the main preprocessing decisions.
findings = [
    'Subject and body were combined into one ticket_text field.',
    f'{len(frame.attrs.get("metadata_columns"))} safe metadata fields were added as TF-IDF tokens.',
    f'Excluded leakage columns: {", ".join(frame.attrs.get("leakage_columns"))}.',
    f'The processed dataset contains {len(frame):,} rows and {frame["model_text"].nunique():,} unique model-text values.',
    'Queue is used as the routing/category target and priority is used as the urgency target.',
]

for item in findings:
    print('-', item)

- Subject and body were combined into one ticket_text field.
- 11 safe metadata fields were added as TF-IDF tokens.
- Excluded leakage columns: answer.
- The processed dataset contains 44,275 rows and 44,273 unique model-text values.
- Queue is used as the routing/category target and priority is used as the urgency target.
